# Theft Detection Pipeline (Anomaly Detection)
This notebook builds a Machine Learning pipeline to identify potential power theft or meter tampering using an **Isolation Forest** (an unsupervised anomaly detection model).

## 1. Import the Data
We will generate synthetic smart meter data (Voltage, Current, Power Factor) and intentionally inject some 'anomalous' events (e.g., sudden current spikes with voltage drops).

In [ ]:
import pandas as pd
import numpy as np

# Generate 30 days of hourly smart meter readings
np.random.seed(42)
timestamps = pd.date_range(start='2023-01-01', periods=720, freq='H')

# Normal operations
voltage = np.random.normal(230, 2, size=720) # 230V standard
current = np.random.normal(15, 3, size=720) + np.sin(np.arange(720) * 2 * np.pi / 24) * 5
power_factor = np.random.uniform(0.85, 0.99, size=720)

df = pd.DataFrame({'Timestamp': timestamps, 'Voltage': voltage, 'Current': current, 'Power_Factor': power_factor})
df['Is_Theft'] = 0 # Ground truth (0 = normal)

# Inject Anomalies (Theft events)
# Anomaly profile: High current, low voltage, poor power factor
anomaly_indices = np.random.choice(720, size=15, replace=False)
df.loc[anomaly_indices, 'Voltage'] = np.random.normal(200, 5, size=15)
df.loc[anomaly_indices, 'Current'] = np.random.normal(40, 5, size=15)
df.loc[anomaly_indices, 'Power_Factor'] = np.random.uniform(0.6, 0.7, size=15)
df.loc[anomaly_indices, 'Is_Theft'] = 1 # Ground truth (1 = anomaly)

df.set_index('Timestamp', inplace=True)
print(f"Dataset shape: {df.shape}")
print(f"Number of Anomalies: {df['Is_Theft'].sum()}")
df.head()

## 2. Clean and Scale the Data
Machine learning models like Isolation Forest operate on distance/density, so it's best practice to scale the features so they have comparable ranges.

In [ ]:
from sklearn.preprocessing import StandardScaler

features = ['Voltage', 'Current', 'Power_Factor']
X = df[features]
y = df['Is_Theft'] # We keep this purely for evaluation later

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=features, index=df.index)

X_scaled_df.head()

## 3. Split the Data into Training/Test Sets
In unsupervised anomaly detection, we train the model on the data (ideally uncontaminated or slightly contaminated data) so it learns 'normal'. We'll split the dataset sequentially.

In [ ]:
from sklearn.model_selection import train_test_split

# We won't pass 'y' to the model during training, as it is unsupervised
X_train, X_test, y_train, y_test = train_test_split(X_scaled_df, y, test_size=0.3, random_state=42, shuffle=False)

print(f"Training Set: {X_train.shape[0]} samples")
print(f"Testing Set: {X_test.shape[0]} samples")

## 4. Create a Model
We initialize an **Isolation Forest**. The `contamination` parameter is our estimate of the percentage of anomalies in the dataset.

In [ ]:
from sklearn.ensemble import IsolationForest

# Assuming roughly 2% of the data might be anomalous
model = IsolationForest(n_estimators=100, contamination=0.02, random_state=42)
print(model)

## 5. Train the Model
Fit the model on the training set to map out the typical behavior bounds.

In [ ]:
model.fit(X_train)
print("Model training complete.")

## 6. Make Predictions
Isolation Forest returns `1` for normal data points and `-1` for anomalies. We will map `-1` to `1` (Theft) and `1` to `0` (Normal) to match our ground truth labels.

In [ ]:
predictions = model.predict(X_test)

# Map: -1 (anomaly) -> 1 (Theft), 1 (normal) -> 0 (Normal)
y_pred = [1 if x == -1 else 0 for x in predictions]

results_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
print(results_df.value_counts())

## 7. Evaluate and Improve
We calculate the Precision, Recall, and F1-score, and plot the detected anomalies on a time series chart.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print("Classification Report:")
print(classification_report(y_test, y_pred))

plt.figure(figsize=(15, 6))

# Plot Original Current data for the Test set
plt.plot(df.index[X_train.shape[0]:], df['Current'].iloc[X_train.shape[0]:], label='Current Draw (A)', color='blue', alpha=0.6)

# Highlight true anomalies
true_anomalies = df.iloc[X_train.shape[0]:][y_test == 1]
plt.scatter(true_anomalies.index, true_anomalies['Current'], color='orange', label='Actual Anomalies', s=100, marker='x')

# Highlight predicted anomalies
pred_anomalies = results_df[results_df['Predicted'] == 1]
plt.scatter(pred_anomalies.index, df['Current'].loc[pred_anomalies.index], color='red', label='Predicted Anomalies', s=50, marker='o')

plt.title('Theft Detection: Current Draw with Detected Anomalies (Test Set)')
plt.xlabel('Timestamp')
plt.ylabel('Current (A)')
plt.legend()
plt.grid(True)
plt.show()